<div dir="rtl" style="text-align:right">
<h1>اگر آینده را باز بگذاریم چه می‌شود؟</h1><p style="text-align:right"><b>پرسش آزمایش:</b> آیا تغییر آینده می‌تواند خروجیِ گذشته را عوض کند؟</p><p style="text-align:right">پیش‌نیاز: <a href="http://127.0.0.1:8000/part-05/chapter-03/33-mask.html"><bdi dir="ltr">33-mask</bdi></a>، <a href="http://127.0.0.1:8000/part-05/chapter-03/34-causal-test.html"><bdi dir="ltr">34-causal-test</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای تکرار پاک، Kernel را Restart و سپس Run All کنید. لینک درس با سروکردن کتاب روی پورت ۸۰۰۰ کار می‌کند؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">این بار از CausalSelfAttention واقعی Mini-GPT استفاده می‌کنیم. وزن‌ها ثابت و آموزش‌ندیده‌اند، Dropout صفر است. آزمون دربارهٔ مسیر اطلاعات است، نه کیفیت زبان. پیش از اجرا حدس بزنید با تغییر دو موقعیت آخر، دو خروجی نخست در کدام حالت باید ثابت بمانند.</p>
</div>

In [ ]:
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
config = ModelConfig(vocab_size=8, context_length=6, embedding_dim=8,
                     num_heads=2, num_layers=1, dropout=0.)
attention = CausalSelfAttention(config).eval()
x = torch.randn(1,4,8)
changed = x.clone()
changed[:,2:] += 5 * torch.randn_like(changed[:,2:])
results = {}
for causal in (False,True):
    trace = {}
    with torch.no_grad():
        original = attention(x, causal=causal, trace=trace)
        modified = attention(changed, causal=causal)
    difference = (original[:,:2]-modified[:,:2]).abs().max().item()
    print("causal:",causal,"prefix maximum change:",difference)
    inspect("actual allowed mask",trace["mask"])
    print(trace["mask"][0,0])
    results[causal] = trace
    if causal:
        torch.testing.assert_close(original[:,:2],modified[:,:2],rtol=0,atol=1e-7)
    else:
        assert difference > 1e-5


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(8,3))
for ax, causal in zip(axes,(False,True)):
    weights = results[causal]["weights"][0,0]
    ax.imshow(weights, vmin=0,vmax=1,cmap="Blues")
    ax.set(title=f"Causal = {causal}",xlabel="Key position",ylabel="Query position",
           xticks=range(4),yticks=range(4))
plt.tight_layout()
plt.show()
allowed = results[True]["mask"]
scores = results[True]["scaled_scores"]
wrong_weights = scores.masked_fill(~allowed, 0).softmax(-1)
print("Forbidden weight when scores are zeroed:", wrong_weights[0,0,0,1:].sum().item())
assert wrong_weights[0,0,0,1:].sum() > 0


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>انتظار و تمرین:</b> با Mask، خانه‌های آینده دقیقاً وزن صفر دارند. بدون آن، هنگام آموزش شیفت‌خورده، ورودیِ موقعیت بعد همان Target فعلی است؛ راهی برای نشت پاسخ باز می‌شود. پایین‌آمدن Loss در این حالت شاهد مدل علّی بهتر نیست. به‌جای صفرکردن امتیازها، چرا منفی بی‌نهایت می‌گذاریم؟ آزمایش را با T=6 بازنویسی کنید و مرز پیشوندِ ثابت را مشخص نگه دارید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2>برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی، مشاهده و دلیل اختلافشان را در یادداشت خود بنویسید. سپس به <a href="http://127.0.0.1:8000/part-05/chapter-03/34-causal-test.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>